<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - `examples.py` simulation

## Path Constants

Define the project name.

In [1]:
PROJECT = "cl_st1_ph1_leticia"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
MEANS_PATTERN = f"sas/output_{PROJECT}/means_group_f{{dim}}.tsv"
FILE_IDS_PATH = Path("file_ids.txt")

## Load factor scores

In [3]:
import pandas as pd

scores_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_df = scores_df.rename(columns={"filename": "file_id"})

scores_df

,file_id,source,model,prompt,group,fac1,fac2,fac3,fac4,fac5,...,v000191,v000192,v000193,v000194,v000195,v000196,v000197,v000198,v000199,v000200
0,t000001,human,human,human,human,-1,-1,3,0,0,...,0,0,1,1,0,0,0,0,0,0
1,t000002,human,human,human,human,0,0,3,1,0,...,0,0,1,1,0,0,0,0,1,0
2,t000003,human,human,human,human,-1,1,3,1,1,...,0,0,1,0,0,0,0,0,1,0
3,t000004,human,human,human,human,0,-1,4,1,0,...,0,0,1,0,0,0,0,0,1,0
4,t000005,human,human,human,human,-4,2,5,1,-1,...,0,0,0,1,1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5110,t005111,ai,gpt,unprofiled,unprofiled_gpt,-4,2,1,1,0,...,0,0,0,0,0,0,0,0,1,0
5111,t005112,ai,gpt,unprofiled,unprofiled_gpt,-3,-1,2,-1,1,...,0,0,0,0,0,0,0,0,0,0
5112,t005113,ai,gpt,unprofiled,unprofiled_gpt,-1,3,1,-2,-1,...,0,0,1,0,0,1,0,0,1,0
5113,t005114,ai,gpt,unprofiled,unprofiled_gpt,-2,0,3,1,0,...,0,0,0,0,0,0,0,0,1,0


## Load file ID mapping

In [4]:
file_ids_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["file_id", "group_filename"],
)

file_ids_df.head()

,file_id,group_filename
0,t000001,human/t000001_human.txt
1,t000002,human/t000002_human.txt
2,t000003,human/t000003_human.txt
3,t000004,human/t000004_human.txt
4,t000005,human/t000005_human.txt


## Merge factor scores with file ID mapping

In [5]:
scores_file_ids_df = scores_df.merge(file_ids_df, on="file_id", how="left")
scores_file_ids_df = scores_file_ids_df[
    ["file_id", "group_filename"]
    + [col for col in scores_file_ids_df.columns if col not in ["file_id", "group_filename"]]
    ]

scores_file_ids_df

,file_id,group_filename,source,model,prompt,group,fac1,fac2,fac3,fac4,...,v000191,v000192,v000193,v000194,v000195,v000196,v000197,v000198,v000199,v000200
0,t000001,human/t000001_human.txt,human,human,human,human,-1,-1,3,0,...,0,0,1,1,0,0,0,0,0,0
1,t000002,human/t000002_human.txt,human,human,human,human,0,0,3,1,...,0,0,1,1,0,0,0,0,1,0
2,t000003,human/t000003_human.txt,human,human,human,human,-1,1,3,1,...,0,0,1,0,0,0,0,0,1,0
3,t000004,human/t000004_human.txt,human,human,human,human,0,-1,4,1,...,0,0,1,0,0,0,0,0,1,0
4,t000005,human/t000005_human.txt,human,human,human,human,-4,2,5,1,...,0,0,0,1,1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5110,t005111,unprofiled_gpt/t001701_gpt.txt,ai,gpt,unprofiled,unprofiled_gpt,-4,2,1,1,...,0,0,0,0,0,0,0,0,1,0
5111,t005112,unprofiled_gpt/t001702_gpt.txt,ai,gpt,unprofiled,unprofiled_gpt,-3,-1,2,-1,...,0,0,0,0,0,0,0,0,0,0
5112,t005113,unprofiled_gpt/t001703_gpt.txt,ai,gpt,unprofiled,unprofiled_gpt,-1,3,1,-2,...,0,0,1,0,0,1,0,0,1,0
5113,t005114,unprofiled_gpt/t001704_gpt.txt,ai,gpt,unprofiled,unprofiled_gpt,-2,0,3,1,...,0,0,0,0,0,0,0,0,1,0


## Load group means

In [7]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

means_dfs = {}

for factor_col in factor_cols:
    fac_num = int(factor_col.replace("fac", ""))
    means_file = Path(MEANS_PATTERN.format(dim=fac_num))

    if not means_file.exists():
        raise FileNotFoundError(f"Means file not found: {means_file}")

    means_df = pd.read_csv(means_file, sep="\t")

    mean_col = f"Mean fac{fac_num}"

    if "group" not in means_df.columns:
        raise ValueError(f"'group' column missing from means file: {means_file}")

    if mean_col not in means_df.columns:
        raise ValueError(f"'{mean_col}' column missing from means file: {means_file}")

    means_df["group"] = means_df["group"].astype(str).str.strip()

    means_dfs[factor_col] = means_df

means_dfs

{'fac1':   Effect           group     N  Mean fac1   SD fac1
 0  group           human  1705  -1.269795  1.056473
 1  group    profiled_gpt  1705  -2.572434  1.521232
 2  group  unprofiled_gpt  1705  -2.527273  1.541481,
 'fac2':   Effect           group     N  Mean fac2   SD fac2
 0  group           human  1705   0.239296  1.370884
 1  group    profiled_gpt  1705   0.667449  1.815499
 2  group  unprofiled_gpt  1705   0.631672  1.742975,
 'fac3':   Effect           group     N  Mean fac3   SD fac3
 0  group           human  1705   3.381818  1.616094
 1  group    profiled_gpt  1705   1.414076  1.168257
 2  group  unprofiled_gpt  1705   1.392375  1.195281,
 'fac4':   Effect           group     N  Mean fac4   SD fac4
 0  group           human  1705   0.880938  0.941614
 1  group    profiled_gpt  1705   0.096774  0.918968
 2  group  unprofiled_gpt  1705   0.180059  0.895909,
 'fac5':   Effect           group     N  Mean fac5   SD fac5
 0  group           human  1705   0.563636  1.010318
 1

## Simulate selection process

### List factors

In [8]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

print("\n".join(
    f"{'' if i == 0 else '#'}FACTOR = {factor!r}"
    for i, factor in enumerate(factor_cols)
))

FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'
#FACTOR = 'fac5'
#FACTOR = 'fac6'


### Copy/Paste the factors list and Uncomment/Comment factors and poles accordingly

In [18]:
FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'
#FACTOR = 'fac5'
#FACTOR = 'fac6'

POLE = "positive"
#POLE = "negative"

sorted_means_dfs = means_dfs[FACTOR].sort_values(f"Mean {FACTOR}", ascending=POLE == "negative")

display(sorted_means_dfs)

groups_in_order = sorted_means_dfs["group"].tolist()

print("\n".join(
    f"{'' if i == 0 else '#'}GROUP = {group!r}"
    for i, group in enumerate(groups_in_order)
))

,Effect,group,N,Mean fac1,SD fac1
0,group,human,1705,-1.269795,1.056473
2,group,unprofiled_gpt,1705,-2.527273,1.541481
1,group,profiled_gpt,1705,-2.572434,1.521232


GROUP = 'human'
#GROUP = 'unprofiled_gpt'
#GROUP = 'profiled_gpt'


### Copy/Paste the groups list and Uncomment/Comment groups accordingly

In `examples.py`, texts with a factor score of exactly `0` are skipped. In this simulation tool, they are not, for the sake of simplicity and to allow for a broader view.

In [10]:
GROUP = 'human'
#GROUP = 'unprofiled_gpt'
#GROUP = 'profiled_gpt'

N_EXAMPLES = 40 # Define the number of examples to select

group_df = scores_file_ids_df.loc[scores_file_ids_df["group"].eq(GROUP)]

if POLE == "positive":
    examples_df = group_df.nlargest(N_EXAMPLES, FACTOR)
elif POLE == "negative":
    examples_df = group_df.nsmallest(N_EXAMPLES, FACTOR)
else:
    raise ValueError(f"Unknown POLE: {POLE!r}")

examples_df = examples_df[["file_id", "group_filename", "group"] + factor_cols]

examples_df

,file_id,group_filename,group,fac1,fac2,fac3,fac4,fac5,fac6
19,t000020,human/t000020_human.txt,human,1,0,6,1,0,0
321,t000322,human/t000322_human.txt,human,1,0,2,0,0,0
347,t000348,human/t000348_human.txt,human,1,2,4,0,0,0
349,t000350,human/t000350_human.txt,human,1,2,2,0,0,0
568,t000569,human/t000569_human.txt,human,1,0,3,1,0,0
973,t000974,human/t000974_human.txt,human,1,2,1,-2,0,1
1497,t001498,human/t001498_human.txt,human,1,2,3,0,0,0
1522,t001523,human/t001523_human.txt,human,1,1,-1,1,0,0
1,t000002,human/t000002_human.txt,human,0,0,3,1,0,0
3,t000004,human/t000004_human.txt,human,0,-1,4,1,0,0


## Fetch information from `examples/`

Write a Jupyter Notebook cell that parses generated LaTeX example files under `examples/` and builds a pandas DataFrame.

Requirements:

- Search only immediate subdirectories of `examples/` whose names match `f<n>_<pos|neg>`.
- Sort subdirectories by numeric factor number, then by pole order: `pos`, `neg`.
- In each matching subdirectory, parse only `.tex` files whose names match `f<n>_<pos|neg>_<m>.tex`, where the factor and pole match the parent directory.
- Sort files by numeric sequence `<m>`.
- For each file, read the first line that starts with `\begin{textsample}{`.
- Extract metadata from titles formatted as:

      POS Dim 1 – human – Score 101.00 – t451\_human.txt

  or:

      NEG Dim 2 – persona_gpt – Score -3.45 – t123\_gpt.txt

- Convert:
  - `POS` to `pole = "positive"`
  - `NEG` to `pole = "negative"`
  - `Dim <n>` to `factor = "fac<n>"`
  - escaped LaTeX underscores, such as `\_`, to `_` in `group` and `group_filename`
  - `Score <n>` to numeric `score`

- Create a pandas DataFrame with exactly these columns:

      example_filename
      factor
      pole
      group
      group_filename
      score

- The DataFrame should be sorted in the order of subdirectory/file parsing.
- Raise clear errors for matching files that lack a `\begin{textsample}{...}` line or whose metadata cannot be parsed.
- After creating the DataFrame, export it to:
  - `examples/examples_summary.ndjson` as newline-delimited JSON, preserving non-ASCII characters.
  - `examples/examples_summary.xlsx` as an Excel file without the DataFrame index.
- Print a confirmation message for each exported file, including the number of records written.

In [11]:
import re
from pathlib import Path

import pandas as pd

EXAMPLES_DIR = Path("examples")

dir_pattern = re.compile(r"^f(?P<factor_num>\d+)_(?P<pole_code>pos|neg)$")
file_pattern_template = r"^f{factor_num}_{pole_code}_(?P<seq>\d+)\.tex$"

textsample_prefix = r"\begin{textsample}{"
metadata_pattern = re.compile(
    r"""^\\begin\{textsample\}\{
        (?P<pole_label>POS|NEG)
        \s+Dim\s+
        (?P<dim>\d+)
        \s+–\s+
        (?P<group>.+?)
        \s+–\s+
        Score\s+
        (?P<score>[+-]?(?:\d+(?:\.\d*)?|\.\d+))
        \s+–\s+
        (?P<group_filename>.+?)
        \}
    """,
    re.VERBOSE,
)

pole_name = {
    "POS": "positive",
    "NEG": "negative",
}

pole_order = {
    "pos": 0,
    "neg": 1,
}

rows = []

if not EXAMPLES_DIR.exists():
    raise FileNotFoundError(f"Examples directory not found: {EXAMPLES_DIR}")

example_dirs = []

for path in EXAMPLES_DIR.iterdir():
    if not path.is_dir():
        continue

    dir_match = dir_pattern.fullmatch(path.name)
    if not dir_match:
        continue

    example_dirs.append(
        {
            "path": path,
            "factor_num": int(dir_match.group("factor_num")),
            "pole_code": dir_match.group("pole_code"),
        }
    )

example_dirs = sorted(
    example_dirs,
    key=lambda item: (item["factor_num"], pole_order[item["pole_code"]]),
)

for example_dir in example_dirs:
    factor_num = example_dir["factor_num"]
    pole_code = example_dir["pole_code"]
    dir_path = example_dir["path"]

    file_pattern = re.compile(
        file_pattern_template.format(
            factor_num=factor_num,
            pole_code=pole_code,
        )
    )

    tex_files = []

    for tex_path in dir_path.iterdir():
        if not tex_path.is_file():
            continue

        file_match = file_pattern.fullmatch(tex_path.name)
        if not file_match:
            continue

        tex_files.append(
            {
                "path": tex_path,
                "seq": int(file_match.group("seq")),
            }
        )

    tex_files = sorted(tex_files, key=lambda item: item["seq"])

    for tex_file in tex_files:
        tex_path = tex_file["path"]

        textsample_line = None

        with open(tex_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith(textsample_prefix):
                    textsample_line = line
                    break

        if textsample_line is None:
            raise ValueError(
                f"No line beginning with {textsample_prefix!r} found in {tex_path}"
            )

        metadata_match = metadata_pattern.match(textsample_line)

        if not metadata_match:
            raise ValueError(
                "Could not parse textsample metadata in "
                f"{tex_path}:\n{textsample_line}"
            )

        pole_label = metadata_match.group("pole_label")
        dim = int(metadata_match.group("dim"))

        if dim != factor_num:
            raise ValueError(
                f"Factor mismatch in {tex_path}: "
                f"directory says f{factor_num}, metadata says Dim {dim}"
            )

        expected_pole_label = pole_code.upper()
        if pole_label != expected_pole_label:
            raise ValueError(
                f"Pole mismatch in {tex_path}: "
                f"directory says {pole_code}, metadata says {pole_label}"
            )

        group = metadata_match.group("group").replace(r"\_", "_").strip()
        group_filename = metadata_match.group("group_filename").replace(r"\_", "_").strip()
        score = float(metadata_match.group("score"))

        rows.append(
            {
                "example_filename": tex_path.name,
                "factor": f"fac{dim}",
                "pole": pole_name[pole_label],
                "group": group,
                "group_filename": group_filename,
                "score": score,
            }
        )

examples_summary_df = pd.DataFrame(
    rows,
    columns=[
        "example_filename",
        "factor",
        "pole",
        "group",
        "group_filename",
        "score",
    ],
)

# Export to NDJSON and Excel
from pathlib import Path

NDJSON_OUT = Path("examples/examples_summary.ndjson")
EXCEL_OUT = Path("examples/examples_summary.xlsx")

examples_summary_df.to_json(
    NDJSON_OUT,
    orient="records",
    lines=True,
    force_ascii=False,
)

examples_summary_df.to_excel(
    EXCEL_OUT,
    index=False,
)

print(f"Wrote {len(examples_summary_df):,} records to {NDJSON_OUT}")
print(f"Wrote {len(examples_summary_df):,} records to {EXCEL_OUT}")

examples_summary_df

Wrote 480 records to examples/examples_summary.ndjson
Wrote 480 records to examples/examples_summary.xlsx


,example_filename,factor,pole,group,group_filename,score
0,f1_pos_001.tex,fac1,positive,human,human/t001523_human.txt,1.0
1,f1_pos_002.tex,fac1,positive,human,human/t001498_human.txt,1.0
2,f1_pos_003.tex,fac1,positive,human,human/t000974_human.txt,1.0
3,f1_pos_004.tex,fac1,positive,human,human/t000322_human.txt,1.0
4,f1_pos_005.tex,fac1,positive,human,human/t000348_human.txt,1.0
...,...,...,...,...,...,...
475,f6_neg_036.tex,fac6,negative,human,human/t000133_human.txt,-2.0
476,f6_neg_037.tex,fac6,negative,human,human/t001368_human.txt,-2.0
477,f6_neg_038.tex,fac6,negative,human,human/t000431_human.txt,-2.0
478,f6_neg_039.tex,fac6,negative,human,human/t001212_human.txt,-1.0
